# Mini-TP 3 — Sirve tu modelo por gRPC (starter)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual, esta semana**

Expón **tu propio modelo** (el mismo que serviste por REST en la Sesión 1) como un servicio **gRPC**. Este notebook es un *starter*: completa las celdas marcadas con `# TODO`.

**Qué debes entregar**
1. Un `scoring.proto` con un servicio para tu modelo (mensajes de entrada y salida **tipados**).
2. Los *stubs* generados y un **servidor** que cargue tu modelo **una sola vez**.
3. Un **cliente** que llame al servicio (**unary**).
4. Un método de **server-streaming** que puntúe un lote.
5. Una **comparación de latencia** contra tu endpoint REST de la Sesión 1, con una breve reflexión.

**Se evalúa:** que corra de punta a punta; que el `.proto` tipe entrada y salida; que funcionen unary y streaming; y la reflexión gRPC vs REST.

> Apóyate en `grpc_tutorial.ipynb`, que hace todo esto paso a paso. Entorno con uv:
> `uv add grpcio grpcio-tools scikit-learn joblib requests` (y lo que use tu modelo).

In [ ]:
# Dependencias (equivalente uv: uv add grpcio grpcio-tools joblib ...)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "grpcio", "grpcio-tools", "scikit-learn", "joblib", "requests"])
print("Listo.")

## 1. Tu modelo

Carga aquí **tu** modelo entrenado (por ejemplo `model.pkl`). Si todavía no lo tienes a mano, entrena uno rápido para probar la mecánica.

In [ ]:
import joblib
# TODO: carga tu modelo. Debe exponer .predict(X)
# modelo = joblib.load("model.pkl")

# --- placeholder para probar el flujo (reemplázalo por tu modelo) ---
from sklearn.linear_model import LogisticRegression
import numpy as np
_X = np.random.RandomState(0).randn(200, 3); _y = (_X.sum(1) > 0).astype(int)
modelo = LogisticRegression().fit(_X, _y)
N_FEATURES = 3  # TODO: ajusta al número de features de tu modelo
print("Modelo cargado. n_features =", N_FEATURES)

## 2. El contrato `.proto`

Define el servicio para tu modelo. Debe tener **al menos** un método unary y uno de streaming.

In [ ]:
%%writefile scoring.proto
syntax = "proto3";
package scoring;

// TODO: ajusta los mensajes a la entrada/salida de TU modelo
message Features {
  repeated double values = 1;
  string model = 2;
}
message Prediction {
  double score = 1;
  string model_version = 2;
}

service Scoring {
  rpc Predict(Features) returns (Prediction);
  // TODO: agrega el método de server-streaming
  // rpc PredictStream(Features) returns (stream Prediction);
}

## 3. Genera los *stubs*

In [ ]:
import sys, subprocess, os
subprocess.run([sys.executable, "-m", "grpc_tools.protoc",
                "-I.", "--python_out=.", "--grpc_python_out=.", "scoring.proto"], check=True)
print("Generados:", [f for f in os.listdir('.') if f.startswith('scoring_pb2')])

## 4. El servidor

Implementa los métodos. Recuerda: el modelo se carga **una vez** (ya lo cargaste arriba).

In [ ]:
import grpc, threading
from concurrent import futures
import scoring_pb2, scoring_pb2_grpc

class ScoringServicer(scoring_pb2_grpc.ScoringServicer):
    def Predict(self, request, context):
        # TODO: usa tu modelo para producir el score
        y = modelo.predict([list(request.values)])[0]
        return scoring_pb2.Prediction(score=float(y), model_version="v1")

    # TODO: implementa PredictStream (recuerda declararlo en el .proto y regenerar)
    # def PredictStream(self, request, context):
    #     for ...:
    #         yield scoring_pb2.Prediction(score=..., model_version="v1")

servidor = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
scoring_pb2_grpc.add_ScoringServicer_to_server(ScoringServicer(), servidor)
servidor.add_insecure_port("[::]:50051")
servidor.start()
print("Servidor gRPC en localhost:50051")

## 5. El cliente (unary)

In [ ]:
canal = grpc.insecure_channel("localhost:50051")
stub = scoring_pb2_grpc.ScoringStub(canal)

# TODO: arma una entrada válida para tu modelo (N_FEATURES valores)
entrada = scoring_pb2.Features(values=[0.2, 1.3, 0.7], model="mi_modelo")
r = stub.Predict(entrada)
print("score:", r.score, "| version:", r.model_version)

## 6. Streaming

Una vez que agregaste `PredictStream` al `.proto`, lo regeneraste y lo implementaste en el servidor, pruébalo aquí.

In [ ]:
# TODO: descomenta cuando tengas PredictStream implementado
# for p in stub.PredictStream(entrada):
#     print(round(p.score, 3))
print("Pendiente: implementar y probar el streaming.")

## 7. Comparación de latencia gRPC vs REST

Levanta (o apunta a) tu REST de la Sesión 1 y mide el tiempo medio por llamada. Anota abajo tu reflexión.

In [ ]:
import time
N = 200
feat = scoring_pb2.Features(values=[0.2, 1.3, 0.7])
t = time.perf_counter()
for _ in range(N):
    stub.Predict(feat)
grpc_ms = (time.perf_counter() - t) / N * 1000
print(f"gRPC: {grpc_ms:.3f} ms/llamada")

# TODO: mide tu endpoint REST (import requests; requests.post(TU_URL, json=...))
# rest_ms = ...
# print(f"REST: {rest_ms:.3f} ms/llamada  ->  gRPC ~{rest_ms/grpc_ms:.1f}x")

### Reflexión (completa)

_TODO: 3–5 líneas. ¿Qué diferencia de latencia observaste? ¿Cuándo usarías gRPC y cuándo REST en tu plataforma? ¿Qué costo tiene gRPC (contrato, tooling, no lo habla el navegador directo)?_

In [ ]:
# limpieza
canal.close(); servidor.stop(0)
print("Servidor detenido.")